In [1]:
import torch
from transformers import GPT2LMHeadModel, DataCollatorForLanguageModeling, TrainingArguments, Trainer, set_seed
import pandas as pd
import numpy as np
from datasets import Dataset, load_from_disk, concatenate_datasets
from transformers import GPT2TokenizerFast
from pathlib import Path
import json

In [2]:
out_dir = Path('../data/processed/artifacts/CPT_user_behavior_V1/')

tokenizer = GPT2TokenizerFast.from_pretrained(out_dir / 'tokenizer')

with open(out_dir / 'token_spec.json', 'r', encoding='utf-8') as f:
    token_spec = json.load(f)
    
ds_user_behavior = load_from_disk(out_dir / 'data' / 'behavior')
ds_meta_title = load_from_disk(out_dir / 'data' / 'meta_title')
ds_meta_genre = load_from_disk(out_dir / 'data' / 'meta_genre')
ds_meta_year  = load_from_disk(out_dir / 'data' / 'meta_year')

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [3]:
def sample_len_stats(ds, n=2000):
    m = min(n, len(ds))
    lens = [len(ds[i]['input_ids']) for i in range(m)]
    return {
        'min': int(np.min(lens)),
        'median': int(np.median(lens)),
        'p95': int(np.percentile(lens, 95)),
        'max': int(np.max(lens)),
    }

print('behavior:', sample_len_stats(ds_user_behavior))
print('meta_title:', sample_len_stats(ds_meta_title))
print('meta_genre:', sample_len_stats(ds_meta_genre))
print('meta_year:', sample_len_stats(ds_meta_year))

behavior: {'min': 152, 'median': 728, 'p95': 1008, 'max': 1008}
meta_title: {'min': 15, 'median': 17, 'p95': 22, 'max': 33}
meta_genre: {'min': 17, 'median': 20, 'p95': 25, 'max': 33}
meta_year: {'min': 16, 'median': 16, 'p95': 16, 'max': 16}


In [4]:
seed = 42
set_seed(seed)
ratio_behavior = 0.5

ds_meta_all = concatenate_datasets([ds_meta_title, ds_meta_genre, ds_meta_year]).shuffle(seed=seed)
ds_user_behavior = ds_user_behavior.shuffle(seed=seed)

max_total = int(min(len(ds_user_behavior) / ratio_behavior, len(ds_meta_all) / (1 - ratio_behavior)))

n_user_behavior = int(max_total * ratio_behavior)
n_meta_all = int(max_total * (1 - ratio_behavior))

ds_user_behavior_selected = ds_user_behavior.select(range(n_user_behavior))
ds_meta_all_selected = ds_meta_all.select(range(n_meta_all))

ds_cpt = concatenate_datasets([ds_user_behavior_selected, ds_meta_all_selected]).shuffle(seed=seed)

print('ds_cpt size:', len(ds_cpt), 'behavior:', n_user_behavior, 'meta:', n_meta_all)

ds_cpt size: 12080 behavior: 6040 meta: 6040


In [5]:
split = ds_cpt.train_test_split(test_size=0.02, seed=seed)
ds_train = split['train']
ds_val = split['test']

print('train:', len(ds_train), 'val:', len(ds_val))

train: 11838 val: 242


In [6]:
model = GPT2LMHeadModel.from_pretrained('gpt2')

In [7]:
model.resize_token_embeddings(len(tokenizer))

Embedding(50838, 768)

In [11]:
'''
model = GPT2LMHeadModel.from_pretrained('gpt2')
model.resize_token_embeddings(len(tokenizer))

model.config.eos_token_id = tokenizer.eos_token_id
model.config.bos_token_id = tokenizer.bos_token_id
model.config.pad_token_id = tokenizer.pad_token_id 

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

out_model_dir = out_dir / 'cpt_gpt2_v1'

args = TrainingArguments(
    output_dir=str(out_model_dir),
    overwrite_output_dir=False,
    num_train_epochs=4,
    learning_rate=5e-5,
    warmup_ratio=0.03,
    weight_decay=0.01,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8, 
    eval_strategy='steps',
    save_strategy='steps',
    eval_steps=200,
    save_steps=200,
    logging_steps=50,
    fp16=True,
    report_to='none',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=2,
    
    
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=collator,
    tokenizer=tokenizer,
)

trainer.train()
trainer.save_model(str(out_model_dir))
tokenizer.save_pretrained(str(out_model_dir))
'''

Step,Training Loss,Validation Loss
200,3.687400,3.461343
400,2.659000,2.551873
600,2.354200,2.209431
800,2.144700,1.998043
1000,2.007500,1.862148
1200,1.927300,1.766068
1400,1.874600,1.718720


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


('..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2_v1\\tokenizer_config.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2_v1\\special_tokens_map.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2_v1\\vocab.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2_v1\\merges.txt',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2_v1\\added_tokens.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2_v1\\tokenizer.json')

In [24]:
base_dir = Path('../data/processed/artifacts/CPT_user_behavior_V1/cpt_gpt2_v1')
best_dir = base_dir / 'checkpoint-1400'

tokenizer = GPT2TokenizerFast.from_pretrained(str(base_dir))
model = GPT2LMHeadModel.from_pretrained(str(best_dir))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
model.eval()

model.config.eos_token_id = tokenizer.eos_token_id
model.config.bos_token_id = tokenizer.bos_token_id
model.config.pad_token_id = tokenizer.pad_token_id

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [54]:
def generate_from_prompt(prompt, max_new_tokens=60, sample=False):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)

    with torch.no_grad():
        out = model.generate(
            **inputs, 
            max_new_tokens=max_new_tokens,
            do_sample=sample,
            top_p=0.9,
            temperature=0.25,
            pad_token_id=model.config.pad_token_id,
            eos_token_id=model.config.eos_token_id,
            bos_token_id=model.config.bos_token_id
        )
        

    return tokenizer.decode(out[0])

In [55]:
np.random.seed(10)

#### SID + title

In [56]:
token_spec['notes']['meta_title_template']

'Movie <sid...> has title: {title}'

In [57]:
n = 10

while n > 0:
    i = np.random.randint(low=0, high=len(ds_train))
    title_ids = ds_train[i]['input_ids']
    title_tokens = tokenizer.convert_ids_to_tokens(title_ids)
    true = tokenizer.convert_tokens_to_string(title_tokens)
    if 'title' in true:
        
        colon_pos = title_tokens.index(':')
        prompt_tokens = title_tokens[:colon_pos + 1]
        promt = tokenizer.convert_tokens_to_string(prompt_tokens)
        pred = generate_from_prompt(promt, max_new_tokens=100, sample=True)
        
        print('The original promt:', true)
        print('The GPT2 answer:', pred)
        print('--------------------------------------------------')

        n -= 1
    else:
        continue

The original promt: <bos><title>Movie <sid_0_739><sid_1_325><sid_2_191><sid_3_223><sid_4_46> has title: Broken Arrow</title><eos>
The GPT2 answer: <bos><title>Movie <sid_0_739><sid_1_325><sid_2_191><sid_3_223><sid_4_46> has title: The Last Days of the Condor</title><eos>
--------------------------------------------------
The original promt: <bos><title>Movie <sid_0_992><sid_1_421><sid_2_483><sid_3_150><sid_4_58> has title: Lawn Dogs</title><eos>
The GPT2 answer: <bos><title>Movie <sid_0_992><sid_1_421><sid_2_483><sid_3_150><sid_4_58> has title: The Last Days of Summer</title><eos>
--------------------------------------------------
The original promt: <bos><title>Movie <sid_0_38><sid_1_100><sid_2_66><sid_3_249><sid_4_101> has title: Rough Magic</title><eos>
The GPT2 answer: <bos><title>Movie <sid_0_38><sid_1_100><sid_2_66><sid_3_249><sid_4_101> has title: The Last of the Mohicans</title><eos>
--------------------------------------------------
The original promt: <bos><title>Movie <sid_0

### **After CPT, the model reliably follows the metadata template: it generates a well-formed title segment and closes it correctly. However, it does not consistently map a given SID to the correct movie title, often producing plausible but mismatched titles.**

#### SID + genres

In [58]:
token_spec['notes']['meta_genre_template']

'The genres in movie <sid...> are: {genres}'

In [59]:
n = 10

while n > 0:
    i = np.random.randint(low=0, high=len(ds_train))
    title_ids = ds_train[i]['input_ids']
    title_tokens = tokenizer.convert_ids_to_tokens(title_ids)
    true = tokenizer.convert_tokens_to_string(title_tokens)
    if 'genres' in true:
        
        colon_pos = title_tokens.index(':')
        prompt_tokens = title_tokens[:colon_pos + 1]
        promt = tokenizer.convert_tokens_to_string(prompt_tokens)
        pred = generate_from_prompt(promt, max_new_tokens=100, sample=True)
        
        print('The original promt:', true)
        print('The GPT2 answer:', pred)
        print('--------------------------------------------------')

        n -= 1
    else:
        continue

The original promt: <bos><genres>The genres in movie <sid_0_739><sid_1_575><sid_2_445><sid_3_50><sid_4_120> are:Drama</genres><eos>
The GPT2 answer: <bos><genres>The genres in movie <sid_0_739><sid_1_575><sid_2_445><sid_3_50><sid_4_120> are:Drama</genres><eos>
--------------------------------------------------
The original promt: <bos><genres>The genres in movie <sid_0_540><sid_1_556><sid_2_433><sid_3_31><sid_4_67> are:Drama,War</genres><eos>
The GPT2 answer: <bos><genres>The genres in movie <sid_0_540><sid_1_556><sid_2_433><sid_3_31><sid_4_67> are:Drama</genres><eos>
--------------------------------------------------
The original promt: <bos><genres>The genres in movie <sid_0_230><sid_1_215><sid_2_76><sid_3_108><sid_4_25> are:Action,Adventure,Comedy</genres><eos>
The GPT2 answer: <bos><genres>The genres in movie <sid_0_230><sid_1_215><sid_2_76><sid_3_108><sid_4_25> are:Drama</genres><eos>
--------------------------------------------------
The original promt: <bos><genres>The genres in

### **The model reliably follows the metadata format. However, the generated genres are often only loosely related to the ground truth: it frequently defaults to common combinations like Comedy/Drama or Comedy/Romance, and sometimes predicts clearly mismatched genres. Overall, CPT improved structural correctness, but the SID → exact genre mapping remains weak**

#### SID + year

In [60]:
token_spec['notes']['meta_year_template']

'The movie <sid...> was released in {year}'

In [61]:
n = 15

while n > 0:
    i = np.random.randint(low=0, high=len(ds_train))
    title_ids = ds_train[i]['input_ids']
    title_tokens = tokenizer.convert_ids_to_tokens(title_ids)
    true = tokenizer.convert_tokens_to_string(title_tokens)
    if 'released' in true:

        in_pos = title_tokens.index('Ġin')
        prompt_tokens = title_tokens[:in_pos + 1]
        promt = tokenizer.convert_tokens_to_string(prompt_tokens)
        pred = generate_from_prompt(promt, max_new_tokens=100, sample=True)
        
        print('The original promt:', true)
        print('The GPT2 answer:', pred)
        print('--------------------------------------------------')

        n -= 1
    else:
        continue

The original promt: <bos><year>The movie <sid_0_992><sid_1_650><sid_2_222><sid_3_98><sid_4_115> was released in 1992</year><eos>
The GPT2 answer: <bos><year>The movie <sid_0_992><sid_1_650><sid_2_222><sid_3_98><sid_4_115> was released in 1995</year><eos>
--------------------------------------------------
The original promt: <bos><year>The movie <sid_0_230><sid_1_644><sid_2_198><sid_3_7><sid_4_110> was released in 1977</year><eos>
The GPT2 answer: <bos><year>The movie <sid_0_230><sid_1_644><sid_2_198><sid_3_7><sid_4_110> was released in 1998</year><eos>
--------------------------------------------------
The original promt: <bos><year>The movie <sid_0_540><sid_1_249><sid_2_18><sid_3_155><sid_4_35> was released in 1998</year><eos>
The GPT2 answer: <bos><year>The movie <sid_0_540><sid_1_249><sid_2_18><sid_3_155><sid_4_35> was released in 1995</year><eos>
--------------------------------------------------
The original promt: <bos><year>The movie <sid_0_230><sid_1_633><sid_2_18><sid_3_200><s

### **The model reliably generates a well-formed block, showing strong template learning. However, the predicted release years are often incorrect and tend to collapse to frequent years (1990s), indicating weak SID-to-year grounding**

#### User bahavior data

In [33]:
token_spec['schema_tokens']

['<user>',
 '</user>',
 '<hist>',
 '<e>',
 '</e>',
 '<title>',
 '</title>',
 '<year>',
 '</year>',
 '<genres>',
 '</genres>']

In [64]:
n = 1

while n > 0:
    i = np.random.randint(low=0, high=len(ds_train))
    title_ids = ds_train[i]['input_ids']
    title_tokens = tokenizer.convert_ids_to_tokens(title_ids)
    if '<hist>' in true:

        in_pos = title_tokens.index('<hist>')
        prompt_tokens = title_tokens[:in_pos + 1]
        promt = tokenizer.convert_tokens_to_string(prompt_tokens)
        pred = generate_from_prompt(promt, max_new_tokens=20, sample=True)
        
        print('The GPT2 answer:', pred)
        print('--------------------------------------------------')

        n -= 1
    else:
        continue

The GPT2 answer: <bos><user><gen_M><age_18><occ_4></user><hist><e><sid_0_739><sid_1_908><sid_2_181><sid_3_31><sid_4_67><rat_4></e><e><sid_0_739><sid_1_908><sid_2_383><sid_3_31><sid_4_67><rat_4></e><e><sid_0_739><sid_1_908><sid_2_181>
--------------------------------------------------


### **After CPT, the model can continue a user-history prompt in the correct structured format. Given a prefix ending at <hist>, the model generates repeated event blocks consisting of multi-level SID tokens (<sid_0_*> ... <sid_4_*>) followed by a rating token (<rat_*>). This indicates that CPT successfully taught the model the syntax and sequential pattern of the behavior corpus**

In [65]:
import re




def extract_genres_from_text(text: str) -> set[str]:
    """
    Достаём список жанров из сгенерированного текста.
    Ожидаем формат ... are:Comedy,Drama</genres> ...
    """
    # Берём кусок между 'are:' и '</genres>' (если есть)
    m = re.search(r'are:(.*?)(</genres>|<eos>|$)', text)
    if not m:
        return set()
    raw = m.group(1).strip()

    # Иногда модель может вставить пробелы/лишние символы
    raw = raw.replace(' ', '')
    if raw == '':
        return set()

    parts = [p for p in raw.split(',') if p]
    return set(parts)

def jaccard(a: set[str], b: set[str]) -> float:
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

@torch.no_grad()
def predict_genres_from_prompt(prompt: str, model, tokenizer, device: str, max_new_tokens: int = 30) -> str:
    inputs = tokenizer(prompt, return_tensors='pt').to(device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,  # важно: для метрики лучше детерминированно
        pad_token_id=model.config.pad_token_id,
        eos_token_id=model.config.eos_token_id,
        bos_token_id=model.config.bos_token_id,
    )
    return tokenizer.decode(out[0])

def build_genre_prompt_from_true_example(example_text: str) -> str:
    """
    Из полного true-текста жанрового примера делаем prompt до 'are:' включительно.
    """
    idx = example_text.find('are:')
    if idx == -1:
        return example_text  # fallback
    return example_text[: idx + len('are:')]


# --- 2) Основная функция для mean Jaccard ---

def mean_jaccard_genres(ds_meta_genre, model, tokenizer, device: str, n: int = 200, seed: int = 42):
    rng = np.random.default_rng(seed)
    idxs = rng.integers(low=0, high=len(ds_meta_genre), size=n)

    scores = []
    empty_pred = 0

    for i in idxs:
        ids = ds_meta_genre[int(i)]['input_ids']
        true_text = tokenizer.decode(ids)

        prompt = build_genre_prompt_from_true_example(true_text)
        pred_text = predict_genres_from_prompt(prompt, model, tokenizer, device=device, max_new_tokens=30)

        true_set = extract_genres_from_text(true_text)
        pred_set = extract_genres_from_text(pred_text)

        if len(pred_set) == 0:
            empty_pred += 1

        scores.append(jaccard(true_set, pred_set))

    scores = np.array(scores, dtype=np.float32)
    return {
        'n': n,
        'mean_jaccard': float(scores.mean()),
        'std_jaccard': float(scores.std()),
        'empty_pred_frac': float(empty_pred / n),
        'scores': scores,  # если захочешь гистограмму/квантили
    }


# --- 3) Запуск ---

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
model.eval()

res = mean_jaccard_genres(ds_meta_genre, model, tokenizer, device=device, n=200, seed=0)
res

{'n': 200,
 'mean_jaccard': 0.24558334052562714,
 'std_jaccard': 0.3657088875770569,
 'empty_pred_frac': 0.0,
 'scores': array([0.        , 0.33333334, 0.33333334, 0.        , 0.        ,
        0.33333334, 0.33333334, 0.5       , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.33333334, 0.        ,
        1.        , 0.        , 0.        , 0.33333334, 1.        ,
        1.        , 0.        , 0.        , 0.6666667 , 0.        ,
        1.        , 0.        , 1.        , 0.5       , 0.25      ,
        0.        , 1.        , 0.        , 0.        , 1.        ,
        0.        , 1.        , 0.        , 0.        , 0.25      ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 1.        , 0.        , 0.5       ,
        0.33333334, 1.        , 0.6666667 , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 1.        ,
        0.        , 0.        , 0.        , 0.33333334, 0.      